# Geo Housing with LightAutoML

This notebook demonstrates how to use `LightAutoML` for automating machine learning tasks. We'll apply it to a housing dataset, focusing on exploratory data analysis, feature engineering, and training a regression model.

## Prerequisites

Before running the notebook, ensure you have the following libraries installed.

In [ ]:
!pip install pandas==2.2.3
!pip install requests
!pip install lightautoml --no-deps
!pip install tabicl
!pip install autowoe
!pip install cmaes
!pip install json2html
!pip install poetry-core
!pip install catboost
!pip install optuna

  Using cached json2html-1.3.0.tar.gz (7.0 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for json2html: filename=json2html-1.3.0-py3-none-any.whl size=7665 sha256=cd7a85505bddd35efe5b8f46b2cb796e71c7716bbe17ecac8fd51e7d9a637896
  Stored in directory: /root/.cache/pip/wheels/61/94/55/83683c3c00a3f45cb4da4565208d8bd752133bab99610dba89
Successfully built json2html
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lightautoml 0.4.2 requires catboost>=0.26.1, which is not installed.
lightautoml 0.4.2 requires optuna, which is not installed.
lightautoml 0.4.2 requires poetry-core<2.0.0,>=1.0.0, which is not installed.
lightautoml 0.4.2 requires tabm; python_version >= "3.9", which is not installed.
lightautoml 0.4.2 requires statsmodels<=0.14.0, but you have sta

## 1. Importing Libraries

We begin by importing necessary libraries for data processing, machine learning, and automated machine learning.

In [ ]:
import numpy as np
import pandas as pd
import requests
import pickle
from sklearn.metrics import accuracy_score, classification_report

# LightAutoML
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

/usr/local/lib/python3.13/dist-packages/lightautoml/transformers/text.py:23: UserWarning: 'fasttext' - package isn't installed
  warnings.warn("'fasttext' - package isn't installed")


## 2. Loading Data

We load both the main dataset and an additional dataset for augmentation.

In [ ]:
lat, lon = 55.536757, 37.161013

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "2010-01-01",
    "end_date": "2023-12-31",
    "daily": "precipitation_sum,rain_sum,snowfall_sum",
    "timezone": "Europe/Moscow"
}

print("Загрузка данных из Open-Meteo...")
response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame({
    'date': pd.to_datetime(data['daily']['time']),
    'prcp': data['daily']['precipitation_sum'],
    'rain_sum': data['daily']['rain_sum'],
    'snow_sum': data['daily']['snowfall_sum']
})

df[['prcp', 'rain_sum', 'snow_sum']] = df[['prcp', 'rain_sum', 'snow_sum']].fillna(0)
df = df.sort_values('date').reset_index(drop=True)

print(f"Загружено {len(df)} дней. Период: {df['date'].min()} — {df['date'].max()}")
df.head()

Загрузка данных из Open-Meteo...
Загружено 5113 дней. Период: 2010-01-01 00:00:00 — 2023-12-31 00:00:00


,date,prcp,rain_sum,snow_sum
0,2010-01-01,0.0,0.0,0.0
1,2010-01-02,0.0,0.0,0.0
2,2010-01-03,0.0,0.0,0.0
3,2010-01-04,0.0,0.0,0.0
4,2010-01-05,0.0,0.0,0.0


## 3. Data Augmentation and Preprocessing

In [ ]:
conditions_target = [
    (df['snow_sum'] > 0),
    (df['rain_sum'] > 0) & (df['snow_sum'] == 0)
]
df['target'] = np.select(conditions_target, [2, 1], default=0)

conditions = [
    df['prcp'] == 0,
    (df['prcp'] > 0) & (df['prcp'] <= 2.5),
    (df['prcp'] > 2.5) & (df['prcp'] <= 10.0),
    df['prcp'] > 10.0
]
choices = ['нет осадков', 'мало осадков', 'средние осадки', 'большое кол-во осадков']
df['prcp_category'] = np.select(conditions, choices, default='нет осадков')

df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek

df['target_lag_1'] = df['target'].shift(1)
df['target_lag_2'] = df['target'].shift(2)
df['target_lag_3'] = df['target'].shift(3)

df['prcp_lag_1'] = df['prcp'].shift(1)
df['prcp_cat_lag_1'] = df['prcp_category'].shift(1)
df['prcp_cat_lag_2'] = df['prcp_category'].shift(2)
df['prcp_cat_lag_3'] = df['prcp_category'].shift(3)

df_clean = df.dropna().copy()

cat_features = ['prcp_cat_lag_1', 'prcp_cat_lag_2', 'prcp_cat_lag_3']
for col in cat_features:
    df_clean[col] = df_clean[col].astype('category')

features = [
    'date', 'month', 'day', 'dayofweek',
    'target_lag_1', 'target_lag_2', 'target_lag_3',
    'prcp_lag_1', 'prcp_cat_lag_1', 'prcp_cat_lag_2', 'prcp_cat_lag_3'
]

df_model = df_clean[features + ['target']].copy()

print("Данные для LightAutoML с лаговыми признаками:")
print(df_model.head())
print(f"\nРаспределение классов:\n{df_model['target'].value_counts()}")
print(f"\nРаспределение осадков:\n{df_model['prcp_cat_lag_1'].value_counts()}")

Данные для LightAutoML с лаговыми признаками:
        date  month  day  dayofweek  target_lag_1  target_lag_2  target_lag_3  \
3 2010-01-04      1    4          0           0.0           0.0           0.0   
4 2010-01-05      1    5          1           0.0           0.0           0.0   
5 2010-01-06      1    6          2           0.0           0.0           0.0   
6 2010-01-07      1    7          3           2.0           0.0           0.0   
7 2010-01-08      1    8          4           2.0           2.0           0.0   

   prcp_lag_1  prcp_cat_lag_1 prcp_cat_lag_2 prcp_cat_lag_3  target  
3         0.0     нет осадков    нет осадков    нет осадков       0  
4         0.0     нет осадков    нет осадков    нет осадков       0  
5         0.0     нет осадков    нет осадков    нет осадков       2  
6         0.0     нет осадков    нет осадков    нет осадков       2  
7         7.7  средние осадки    нет осадков    нет осадков       2  

Распределение классов:
target
0    2078
1    1

## 4. Modeling with LightAutoML

In [ ]:
test_size = 365
train = df_model.iloc[:-test_size].copy()
test = df_model.iloc[-test_size:].copy()

print(f"Размер train: {len(train)}, test: {len(test)}")

N_THREADS = 2
N_FOLDS = 5
TIMEOUT = 60 * 10
TARGET_NAME = 'target'

task = Task('multiclass', metric='f1_macro')

roles = {
    'target': TARGET_NAME,
    'date': 'date'
}

automl = TabularAutoML(
    task=task,
    timeout=TIMEOUT,
    cpu_limit=N_THREADS,
    general_params={'use_algos': [['linear_l2', 'lgb', 'lgb_tuned']]},
    reader_params={'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': 42}
)

print("Начинаем обучение LightAutoML (multiclass)...")
oof_pred = automl.fit_predict(train, roles=roles, verbose=1)
print("Обучение завершено")

Размер train: 4745, test: 365
Начинаем обучение LightAutoML (multiclass)...
[16:54:54] Stdout logging level is INFO.


INFO:lightautoml.automl.presets.base:Stdout logging level is INFO.


[16:54:54] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer


[16:54:54] Task: multiclass



INFO:lightautoml.automl.presets.base:Task: multiclass



[16:54:54] Start automl preset with listed constraints:


... (50 outputs left out)

INFO:lightautoml.automl.presets.base:Automl preset training completed in 308.95 seconds



[17:00:03] Model description:
Final prediction for new objects (level 0) = 
	 0.05133 * (5 averaged models Lvl_0_Pipe_0_Mod_0_LinearL2) +
	 0.17205 * (5 averaged models Lvl_0_Pipe_1_Mod_0_LightGBM) +
	 0.77661 * (5 averaged models Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM) 



INFO:lightautoml.automl.presets.base:Model description:
Final prediction for new objects (level 0) = 
	 0.05133 * (5 averaged models Lvl_0_Pipe_0_Mod_0_LinearL2) +
	 0.17205 * (5 averaged models Lvl_0_Pipe_1_Mod_0_LightGBM) +
	 0.77661 * (5 averaged models Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM) 



Обучение завершено


## 5. Making Predictions and Submission

Generate predictions for the test data and save them for submission.

In [ ]:
test_pred = automl.predict(test)

predictions = np.argmax(test_pred.data, axis=1)

acc = accuracy_score(test['target'], predictions)

print(f"LightAutoML Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(test['target'], predictions,
                            target_names=['Сухо (0)', 'Дождь (1)', 'Снег (2)']))

LightAutoML Accuracy: 0.6438

Classification Report:
              precision    recall  f1-score   support

    Сухо (0)       0.54      0.48      0.51       128
   Дождь (1)       0.60      0.67      0.64       125
    Снег (2)       0.79      0.79      0.79       112

    accuracy                           0.64       365
   macro avg       0.65      0.65      0.65       365
weighted avg       0.64      0.64      0.64       365



## Conclusion

This notebook demonstrated how to use `LightAutoML` for a regression task on a housing dataset. The approach included data augmentation, feature engineering, model setup, and submission creation. LightAutoML helps streamline the machine learning process by automatically selecting and optimizing the best models for your data.

In [ ]:
with open('automl_model_multiclass.pkl', 'wb') as f:
    pickle.dump(automl, f)

print("Модель успешно сохранена в файл automl_model_multiclass.pkl")
print(f"Версия numpy при сохранении: {np.__version__}")

Модель успешно сохранена в файл automl_model_multiclass.pkl
Версия numpy при сохранении: 2.1.3
